# 实证研究最常见的 20 个错误

本笔记本由文章自动生成，代码和[原文](https://wxrkz123.github.io/blog/Empirical-Mistakes/)完全一致。文中的解释、公式、图和每段代码的输出，请对照原文阅读。

所有模拟共用下面的准备代码：

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
import warnings
warnings.simplefilter("ignore")

def ols(y, X):
    """最小二乘的系数和常规标准误，用于大量重复的模拟。"""
    X = np.column_stack([np.ones(len(y)), X])
    b, *_ = np.linalg.lstsq(X, y, rcond=None)
    e = y - X @ b
    V = e @ e / (len(y) - X.shape[1]) * np.linalg.inv(X.T @ X)
    return b, np.sqrt(np.diag(V))
print("ok")

### 1. 合并之后不检查行数

一对一的合并，行数不应该变化。如果右表中的键有重复（同一家公司有两条行业记录、同一天有两条价格），合并后左表的行会被复制，之后的求和、平均、回归都被重复计算，而且不会有任何提示：

In [ ]:
firms = pd.DataFrame({"firm": [1, 2, 3], "assets": [100, 200, 300]})
industry = pd.DataFrame({"firm": [1, 2, 2, 3], "ind": ["银行", "电子", "通信", "医药"]})   # 公司 2 有两条记录
merged = firms.merge(industry, on="firm")
print(len(firms), "→", len(merged), "  总资产：", firms["assets"].sum(), "→", merged["assets"].sum())
try:
    firms.merge(industry, on="firm", validate="one_to_one")
except pd.errors.MergeError as e:
    print("validate 参数直接报错：", str(e).splitlines()[0])

### 2. 缺失值让每个回归的样本悄悄变化

统计软件会自动删掉有缺失值的观测。不同的回归用了不同的控制变量，样本就不同，各列之间的系数变化可能来自样本变化，而不是控制变量的作用：

In [ ]:
rng = np.random.default_rng(1)
n = 1000
df = pd.DataFrame({"x": rng.normal(size=n), "z": rng.normal(size=n)})
df["y"] = 1 + 0.5 * df["x"] + rng.normal(size=n)
df.loc[df["y"] > 1.5, "z"] = np.nan                 # 控制变量 z 只在 y 较大时缺失
m1 = smf.ols("y ~ x", data=df).fit()
m2 = smf.ols("y ~ x + z", data=df).fit()
print(f"不加 z：n = {int(m1.nobs)}，x 的系数 {m1.params['x']:.3f}")
print(f"加上 z：n = {int(m2.nobs)}，x 的系数 {m2.params['x']:.3f}")

### 4. 幸存者偏差

只看到“活下来”的样本。如果数据库里只有现在还存在的基金或公司，表现差而倒闭、退市的那些就消失了，平均表现被高估：

In [ ]:
rng = np.random.default_rng(2)
ret = rng.normal(0.0, 0.15, size=(5000, 10))         # 5000 只基金 10 年的年收益，真实平均为 0
wealth = np.cumprod(1 + ret, axis=1)
alive = wealth.min(axis=1) > 0.7                     # 净值曾经跌破 0.7 的基金被清盘
print(f"全部基金的平均年收益：{ret.mean():.2%}")
print(f"活下来的 {alive.mean():.0%} 的基金：{ret[alive].mean():.2%}")

### 6. 遗漏变量偏差

遗漏了一个同时影响被解释变量、又和解释变量相关的变量，系数就吸收了它的作用：

In [ ]:
rng = np.random.default_rng(3)
n = 5000
ability = rng.normal(size=n)
educ = 12 + 2 * ability + rng.normal(size=n)
wage = 1 + 0.08 * educ + 0.2 * ability + rng.normal(0, 0.3, size=n)
print(f"真实的教育回报 0.080；遗漏能力：{ols(wage, educ)[0][1]:.3f}；控制能力：{ols(wage, np.column_stack([educ, ability]))[0][1]:.3f}")

### 7. 坏控制：控制了不该控制的变量

“控制变量越多越好”是错的。控制了处理变量的**结果**（中介变量），会把一部分真实效应控制掉；控制了处理和结果的**共同结果**（对撞变量），会凭空制造出相关：

In [ ]:
rng = np.random.default_rng(4)
n = 5000
talent = rng.normal(size=n)                          # 才华
looks = rng.normal(size=n)                           # 外貌，和才华毫无关系
famous = (talent + looks + rng.normal(size=n) > 1.5).astype(float)   # 出名取决于两者
b_all = ols(looks, talent)[0][1]
b_ctrl = ols(looks, np.column_stack([talent, famous]))[0][1]
b_sub = ols(looks[famous == 1], talent[famous == 1])[0][1]
print(f"全部样本：{b_all:.3f}；控制“是否出名”：{b_ctrl:.3f}；只看名人：{b_sub:.3f}")

### 10. 把 Logit 的系数当成边际效应

非线性模型的系数不能像线性回归那样直接解释：

In [ ]:
rng = np.random.default_rng(5)
n = 5000
x = rng.normal(size=n)
y = (rng.uniform(size=n) < 1 / (1 + np.exp(-(-1 + 0.8 * x)))).astype(int)
logit = sm.Logit(y, sm.add_constant(x)).fit(disp=0)
ame = logit.get_margeff(at="overall").margeff[0]
lpm = sm.OLS(y, sm.add_constant(x)).fit().params[1]
print(f"Logit 系数 {logit.params[1]:.3f}；平均边际效应 {ame:.3f}；线性概率模型 {lpm:.3f}")

### 11. 标准误没有聚类，或者聚类在了错误的层级

处理在省份层面分配（比如某些省份实施了政策），而误差项在同一省份内相关，常规标准误会严重低估不确定性：

In [ ]:
rng = np.random.default_rng(6)
G, m, reps = 50, 40, 500                     # 50 个省份，每省 40 个观测
rej_ols = rej_cl = 0
g = np.repeat(np.arange(G), m)
for _ in range(reps):
    d = np.repeat(rng.integers(0, 2, G), m).astype(float)      # 政策在省份层面分配，真实效应为 0
    y = np.repeat(rng.normal(size=G), m) + rng.normal(size=G * m)
    fit = sm.OLS(y, sm.add_constant(d))
    rej_ols += abs(fit.fit().tvalues[1]) > 1.96
    rej_cl += abs(fit.fit(cov_type="cluster", cov_kwds={"groups": g}).tvalues[1]) > 1.96
print(f"名义水平 5%；常规标准误的拒绝率 {rej_ols / reps:.0%}，按省份聚类 {rej_cl / reps:.0%}")

### 13. “一个显著、一个不显著”不等于两者有显著差异

男性样本中系数显著，女性样本中不显著，于是得出结论“效应存在性别差异”。这是最常见的统计错误之一：

In [ ]:
b1, se1 = 0.25, 0.10          # 男性：t = 2.5，显著
b2, se2 = 0.10, 0.10          # 女性：t = 1.0，不显著
diff, se_diff = b1 - b2, np.sqrt(se1 ** 2 + se2 ** 2)
print(f"差异 {diff:.2f}，标准误 {se_diff:.3f}，t = {diff / se_diff:.2f}")

### 14. 多重检验与 p-hacking

尝试足够多的设定、变量、子样本，总能找到一个显著的结果：

In [ ]:
rng = np.random.default_rng(7)
reps, k, n = 1000, 20, 200
found = 0
for _ in range(reps):
    y = rng.normal(size=n)
    X = rng.normal(size=(n, k))               # 20 个和 y 毫无关系的候选变量
    t = [abs(ols(y, X[:, j])[0][1] / ols(y, X[:, j])[1][1]) for j in range(k)]
    found += max(t) > 1.96
print(f"试 20 个无关变量，至少一个“显著”的概率：{found / reps:.0%}（理论值 1 − 0.95^20 = {1 - 0.95 ** 20:.0%}）")

### 15. 把“不显著”当成“没有效应”

不显著只说明数据不足以排除 0，可能是效应真的很小，也可能只是样本太小、检验功效太低：

In [ ]:
rng = np.random.default_rng(8)
reps = 2000
for n in [50, 200, 1000]:
    sig = 0
    for _ in range(reps):
        d = rng.integers(0, 2, n)
        y = 0.2 * d + rng.normal(size=n)          # 真实效应 0.2 个标准差
        b, se = ols(y, d)
        sig += abs(b[1] / se[1]) > 1.96
    print(f"n = {n:4d}：检验功效 {sig / reps:.0%}")

### 19. 伪回归

两个互相独立的随机游走（比如两个都有趋势的宏观序列）做回归，经常得到高度显著的系数和很高的 R²：

In [ ]:
rng = np.random.default_rng(9)
reps, T = 1000, 200
rej, r2 = 0, []
for _ in range(reps):
    y, x = rng.normal(size=(2, T)).cumsum(axis=1)       # 两个互相独立的随机游走
    b, se = ols(y, x)
    rej += abs(b[1] / se[1]) > 1.96
    r2.append(np.corrcoef(x, y)[0, 1] ** 2)
print(f"两个独立的随机游走：{rej / reps:.0%} 的回归“显著”，R² 的中位数 {np.median(r2):.2f}")

### 20. 数据泄漏与样本内选模型

在全部数据上挑选“最好的”模型或策略，再在同一份数据上报告它的表现，结果一定过于乐观：

In [ ]:
rng = np.random.default_rng(10)
n_strategies, T = 200, 240
rets = rng.normal(0, 0.04, size=(n_strategies, 2 * T))     # 200 个毫无预测能力的策略，各 40 年月收益
ins, oos = rets[:, :T], rets[:, T:]
sharpe = lambda r: r.mean(axis=-1) / r.std(axis=-1) * np.sqrt(12)
best = sharpe(ins).argmax()
print(f"样本内最好的策略：夏普比率 {sharpe(ins)[best]:.2f}；同一个策略在样本外：{sharpe(oos[best]):.2f}")